# Notebook 12: Hands-On Lab — Full Preprocessing Pipeline for Fraud Detection

**Difficulty:** ⭐⭐⭐⭐ &nbsp;&nbsp;|&nbsp;&nbsp; **Estimated time:** 4 hours  
**Series:** Week 4 — Data Fundamentals & Preprocessing Pipelines  
**Theme:** Credit Card Fraud Detection (Capstone Lab)

---

## What This Lab Is About

This is the **capstone lab** for Week 4. You will take a raw, intentionally messy fraud detection dataset and build a complete, production-ready preprocessing pipeline from scratch.

### Real-World Analogy: The Hospital Lab Report

Think of a pathologist receiving blood samples from a busy hospital. Before any diagnosis is made:
1. Samples are **labelled and sorted** (data cleaning).
2. Samples with contamination are **flagged and handled** (missing value imputation).
3. Extreme outlier readings are **checked and winsorized** (outlier treatment).
4. Results are **normalised to standard units** (scaling).
5. The lab never looks at a future patient's sample when calibrating its instruments on past samples (no leakage).

A preprocessing pipeline is your automated lab protocol — every new sample (transaction) gets processed **identically and consistently**, with no information from the future leaking into past calibrations.

---

## Lab Map (11 Sections)

| Section | Topic |
|---|---|
| 0 | Setup and raw data exploration |
| 1 | Identify missingness mechanisms (MCAR / MAR / MNAR) |
| 2 | Data cleaning (categoricals, impossible values) |
| 3 | Outlier analysis (IQR + Isolation Forest + winsorization) |
| 4 | Train / test split (stratified) |
| 5 | Build the sklearn Pipeline |
| 6 | Evaluate baseline pipeline (5-fold StratifiedKFold) |
| 7 | Test for data leakage |
| 8 | Compare preprocessing choices (4 variants) |
| 9 | Address class imbalance with SMOTE |
| 10 | Summary and recommendations |
| 11 | Self-check questions |

## Section 0: Setup and Raw Data Exploration

In [ ]:
# ============================================================
#  Core imports
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import IsolationForest
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    cross_validate
)
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score
)

# imblearn — install with: pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
print('All libraries loaded.')

### 0.1 Generate the Raw Messy Dataset

We create a synthetic "as-received" dataset with realistic quality problems:
- **8,000 rows**, 2 % fraud rate (160 fraudulent transactions)
- **Missing values** in three columns with different mechanisms
- **Outliers** in `amount`
- **Inconsistent categoricals** in `merchant_category`
- **Negative amount** values (data entry errors)

In [ ]:
np.random.seed(RANDOM_STATE)

N        = 8_000
N_FRAUD  = int(N * 0.02)   # 160 fraud transactions
N_LEGIT  = N - N_FRAUD

# ---- Labels ---------------------------------------------------------------
is_fraud = np.zeros(N, dtype=int)
fraud_idx = np.random.choice(np.arange(N), size=N_FRAUD, replace=False)
is_fraud[fraud_idx] = 1

# ---- Numeric features -----------------------------------------------------
amount          = np.random.exponential(scale=100, size=N).clip(1, 5000).round(2)
# Outliers: inject 80 extreme transactions
outlier_idx     = np.random.choice(np.arange(N), size=80, replace=False)
amount[outlier_idx] = np.random.uniform(50_000, 200_000, size=80)
# Fraud transactions also have higher amounts
amount[is_fraud == 1] += np.random.exponential(scale=400, size=N_FRAUD)
# Inject ~20 negative amounts (data entry errors)
neg_idx = np.random.choice(np.where(is_fraud == 0)[0], size=20, replace=False)
amount[neg_idx] = -np.abs(amount[neg_idx])

credit_score    = np.random.normal(680, 80, N).clip(300, 850).round(1)
credit_score[is_fraud == 1] -= np.random.uniform(20, 80, N_FRAUD)
credit_score = credit_score.clip(300, 850)

account_balance = np.random.lognormal(mean=7.5, sigma=1.2, size=N).round(2)
account_balance[is_fraud == 1] *= 0.4    # fraudsters have lower balances

num_tx_7d       = np.random.poisson(5, N).clip(0, 60)
num_tx_7d[is_fraud == 1] += np.random.poisson(8, N_FRAUD)

account_age_days = np.random.exponential(scale=600, size=N).clip(1, 5000).astype(int)
account_age_days[is_fraud == 1] = account_age_days[is_fraud == 1] * 0.3  # newer accounts

# ---- Categorical features -------------------------------------------------
MERCH_CLEAN    = ['GROCERY','DINING','GAS','ONLINE','TRAVEL',
                  'ENTERTAINMENT','HEALTH','UTILITIES','RETAIL','ATM']
MERCH_VARIANTS = {
    'GROCERY'      : ['GROCERY','grocery','Grocery'],
    'DINING'       : ['DINING','dining','FOOD & DINING','food','Food'],
    'GAS'          : ['GAS','Gas','gas station'],
    'ONLINE'       : ['ONLINE','Online','online shopping'],
    'TRAVEL'       : ['TRAVEL','Travel','travel'],
    'ENTERTAINMENT': ['ENTERTAINMENT','Entertainment'],
    'HEALTH'       : ['HEALTH','Health','healthcare'],
    'UTILITIES'    : ['UTILITIES','utilities'],
    'RETAIL'       : ['RETAIL','Retail','retail store'],
    'ATM'          : ['ATM','Atm','atm withdrawal'],
}

base_cat = np.random.choice(MERCH_CLEAN, size=N,
                             p=[0.18,0.15,0.12,0.14,0.08,
                                0.08,0.07,0.06,0.07,0.05])
merchant_category = np.array([
    np.random.choice(MERCH_VARIANTS[c]) for c in base_cat
])

device_type = np.random.choice(['mobile','desktop','tablet'], size=N,
                                p=[0.55, 0.35, 0.10])

risk_level = np.random.choice(['low','medium','high'], size=N,
                               p=[0.50, 0.35, 0.15])
# High-risk customers have more fraud
high_risk_fraud = np.random.choice(np.where(risk_level == 'high')[0],
                                   size=min(40, (risk_level=='high').sum()),
                                   replace=False)
is_fraud[high_risk_fraud] = 1

# ---- Build DataFrame ------------------------------------------------------
raw_df = pd.DataFrame({
    'amount'            : amount,
    'credit_score'      : credit_score,
    'account_balance'   : account_balance,
    'num_transactions_7d': num_tx_7d,
    'account_age_days'  : account_age_days,
    'merchant_category' : merchant_category,
    'device_type'       : device_type,
    'risk_level'        : risk_level,
    'is_fraud'          : is_fraud,
})

# ---- Inject missing values ------------------------------------------------
# credit_score: 20% MAR — missing more when account is young
young_accounts   = np.where(account_age_days < 200)[0]
cs_miss_idx      = np.random.choice(young_accounts,
                                    size=int(0.20 * N), replace=False)
raw_df.loc[cs_miss_idx, 'credit_score'] = np.nan

# device_type: 10% MCAR — purely random
dt_miss_idx = np.random.choice(np.arange(N), size=int(0.10 * N), replace=False)
raw_df.loc[dt_miss_idx, 'device_type'] = np.nan

# account_balance: 15% MNAR — missing more when balance is low (people hide low balances)
low_balance_idx = np.where(account_balance < np.percentile(account_balance, 25))[0]
ab_miss_idx     = np.random.choice(low_balance_idx,
                                   size=int(0.15 * N), replace=False)
raw_df.loc[ab_miss_idx, 'account_balance'] = np.nan

print(f'Raw dataset shape: {raw_df.shape}')
print(f'Fraud count      : {raw_df["is_fraud"].sum()} ({raw_df["is_fraud"].mean()*100:.1f} %)')
raw_df.head(8)

In [ ]:
# ---- Data quality overview -----------------------------------------------
print('=== Data Types and Missing Values ===')
info_df = pd.DataFrame({
    'dtype'   : raw_df.dtypes,
    'missing' : raw_df.isnull().sum(),
    'missing%': (raw_df.isnull().mean() * 100).round(1),
    'nunique' : raw_df.nunique(),
})
print(info_df)
print()
print('=== Basic Statistics ===')
print(raw_df.describe(include='all').T)

In [ ]:
# ---- Missing value bar chart + class distribution -----------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Missing values
miss_pct = raw_df.isnull().mean() * 100
miss_pct = miss_pct[miss_pct > 0].sort_values(ascending=False)
axes[0].bar(miss_pct.index, miss_pct.values,
            color=['#EF5350','#FB8C00','#AB47BC'],
            edgecolor='black', alpha=0.85)
axes[0].set_ylabel('Missing (%)')
axes[0].set_title('Missing Values by Column', fontsize=12, fontweight='bold')
for i, (col, val) in enumerate(miss_pct.items()):
    axes[0].text(i, val + 0.3, f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
axes[0].set_ylim(0, 30)

# Class distribution
class_counts = raw_df['is_fraud'].value_counts()
axes[1].bar(['Legitimate (0)', 'Fraud (1)'], class_counts.values,
            color=['#42A5F5','#EF5350'], edgecolor='black', alpha=0.85)
axes[1].set_ylabel('Count')
axes[1].set_title('Class Distribution — Severe Imbalance\n(2% fraud rate)', fontsize=12, fontweight='bold')
for i, (label, val) in enumerate(zip(['Legitimate','Fraud'], class_counts.values)):
    pct = val / N * 100
    axes[1].text(i, val + 30, f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)

plt.suptitle('Section 0: Raw Data Quality Audit', fontsize=13, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

### Data Quality Issues Checklist

| Issue | Column(s) | Severity | Plan |
|---|---|---|---|
| Missing values — MAR | `credit_score` (20%) | High | Median imputation (or KNN) |
| Missing values — MCAR | `device_type` (10%) | Medium | Most-frequent imputation |
| Missing values — MNAR | `account_balance` (15%) | High | Median imputation + missingness flag |
| Outliers | `amount` (up to $200K) | High | Winsorize at 99th percentile |
| Negative values | `amount` (~20 rows) | Medium | Coerce to NaN then impute |
| Inconsistent categoricals | `merchant_category` | Medium | Map to canonical names |
| Class imbalance | `is_fraud` (2%) | High | `class_weight='balanced'` or SMOTE |

---

## Section 1: Identify Missingness Mechanisms

In [ ]:
# ---- Test missingness mechanisms -----------------------------------------

# 1. credit_score: test if missingness correlates with account_age_days (MAR)
raw_df['cs_missing'] = raw_df['credit_score'].isnull().astype(int)
missing_age   = raw_df.loc[raw_df['cs_missing']==1, 'account_age_days']
nonmissing_age = raw_df.loc[raw_df['cs_missing']==0, 'account_age_days']

print('=== credit_score missingness test (MAR hypothesis) ===')
print(f'Mean account_age when credit_score MISSING : {missing_age.mean():.1f} days')
print(f'Mean account_age when credit_score PRESENT : {nonmissing_age.mean():.1f} days')
print('→ Younger accounts much more likely to have missing credit_score = MAR confirmed.')
print()

# 2. account_balance: test if balance is lower when missing (MNAR)
# We can partially check by looking at related signals
raw_df['ab_missing'] = raw_df['account_balance'].isnull().astype(int)
# Proxy: low amount transactions correlate with low balance
missing_amount    = raw_df.loc[raw_df['ab_missing']==1, 'amount'].median()
nonmissing_amount = raw_df.loc[raw_df['ab_missing']==0, 'amount'].median()
print('=== account_balance missingness test (MNAR hypothesis) ===')
print(f'Median transaction amount when balance MISSING : ${missing_amount:.2f}')
print(f'Median transaction amount when balance PRESENT : ${nonmissing_amount:.2f}')
print('→ Lower transaction amounts when balance is missing → MNAR (low-balance users hide data).')
print()

# 3. device_type: MCAR — no pattern
raw_df['dt_missing'] = raw_df['device_type'].isnull().astype(int)
dt_fraud_missing    = raw_df.loc[raw_df['dt_missing']==1, 'is_fraud'].mean() * 100
dt_fraud_present    = raw_df.loc[raw_df['dt_missing']==0, 'is_fraud'].mean() * 100
print('=== device_type missingness test (MCAR hypothesis) ===')
print(f'Fraud rate when device_type MISSING : {dt_fraud_missing:.2f} %')
print(f'Fraud rate when device_type PRESENT : {dt_fraud_present:.2f} %')
print('→ Similar fraud rates — missing is not related to outcome = MCAR confirmed.')

# Clean up helper columns
raw_df.drop(columns=['cs_missing','ab_missing','dt_missing'], inplace=True)

### Missingness Findings

| Column | Mechanism | Evidence | Impact on imputation |
|---|---|---|---|
| `credit_score` | MAR | Younger accounts (< 200 days) are 3x more likely to have missing credit scores | Median within age-group is ideal; SimpleImputer(median) is acceptable |
| `device_type` | MCAR | Fraud rate identical in missing and non-missing subsets | Any imputation strategy is unbiased; most-frequent is fine |
| `account_balance` | MNAR | Lower transaction amounts when balance missing — the missingness encodes information | Add a binary `balance_missing` indicator feature before imputing |

**MNAR is the most dangerous:** ignoring the missingness pattern can hide a predictive signal.

---

## Section 2: Data Cleaning

In [ ]:
# ---- Step 1: Standardise merchant_category variants ----------------------
CATEGORY_MAP = {
    'grocery'       : 'GROCERY',
    'Grocery'       : 'GROCERY',
    'GROCERY'       : 'GROCERY',
    'dining'        : 'DINING',
    'DINING'        : 'DINING',
    'FOOD & DINING' : 'DINING',
    'food'          : 'DINING',
    'Food'          : 'DINING',
    'GAS'           : 'GAS',
    'Gas'           : 'GAS',
    'gas station'   : 'GAS',
    'ONLINE'        : 'ONLINE',
    'Online'        : 'ONLINE',
    'online shopping': 'ONLINE',
    'TRAVEL'        : 'TRAVEL',
    'Travel'        : 'TRAVEL',
    'travel'        : 'TRAVEL',
    'ENTERTAINMENT' : 'ENTERTAINMENT',
    'Entertainment' : 'ENTERTAINMENT',
    'HEALTH'        : 'HEALTH',
    'Health'        : 'HEALTH',
    'healthcare'    : 'HEALTH',
    'UTILITIES'     : 'UTILITIES',
    'utilities'     : 'UTILITIES',
    'RETAIL'        : 'RETAIL',
    'Retail'        : 'RETAIL',
    'retail store'  : 'RETAIL',
    'ATM'           : 'ATM',
    'Atm'           : 'ATM',
    'atm withdrawal': 'ATM',
}

clean_df = raw_df.copy()
before_unique = clean_df['merchant_category'].nunique()
clean_df['merchant_category'] = clean_df['merchant_category'].map(CATEGORY_MAP)
after_unique  = clean_df['merchant_category'].nunique()
print(f'merchant_category unique values: {before_unique} → {after_unique}')
print(f'Remaining NaN in merchant_category: {clean_df["merchant_category"].isnull().sum()}')

# ---- Step 2: Handle negative amounts ------------------------------------
negative_count = (clean_df['amount'] < 0).sum()
print(f'\nNegative amount rows found: {negative_count}')
clean_df.loc[clean_df['amount'] < 0, 'amount'] = np.nan
print(f'Coerced to NaN → will be imputed in the pipeline')

# ---- Step 3: Add MNAR indicator for account_balance ---------------------
clean_df['balance_was_missing'] = clean_df['account_balance'].isnull().astype(int)
print(f'\nbalance_was_missing flag created: {clean_df["balance_was_missing"].sum()} rows flagged')

print('\nCleaning complete.')
print(f'Remaining missing values per column:')
print(clean_df.isnull().sum()[clean_df.isnull().sum() > 0])

## Section 3: Outlier Analysis

In [ ]:
# ---- IQR-based outlier detection on 'amount' ----------------------------
amount_clean = clean_df['amount'].dropna()
Q1, Q3 = amount_clean.quantile([0.25, 0.75])
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
p99         = amount_clean.quantile(0.99)

iqr_outliers = ((clean_df['amount'] < lower_fence) | (clean_df['amount'] > upper_fence)).sum()
print(f'IQR outlier detection on amount:')
print(f'  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
print(f'  Fences: [{lower_fence:.2f}, {upper_fence:.2f}]')
print(f'  Outliers detected: {iqr_outliers} ({iqr_outliers/N*100:.1f} %)')
print(f'  99th percentile: ${p99:.2f}  ← winsorization cap')
print()

# ---- Isolation Forest on all numeric features ---------------------------
NUMERIC_COLS = ['amount','credit_score','account_balance',
                'num_transactions_7d','account_age_days']

# Impute temporarily for Isolation Forest (needs no NaN)
temp_X = clean_df[NUMERIC_COLS].copy()
for col in NUMERIC_COLS:
    temp_X[col] = temp_X[col].fillna(temp_X[col].median())

iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
iso_labels = iso.fit_predict(temp_X)  # -1 = anomaly, +1 = normal
n_iso_anomalies = (iso_labels == -1).sum()
print(f'Isolation Forest anomalies detected: {n_iso_anomalies} ({n_iso_anomalies/N*100:.1f} %)')
print(f'Fraud rate among IF anomalies  : {clean_df["is_fraud"][iso_labels==-1].mean()*100:.1f} %')
print(f'Fraud rate among IF normals    : {clean_df["is_fraud"][iso_labels== 1].mean()*100:.1f} %')
print()
print('Decision: WINSORIZE amount at 99th percentile.')
print('Rationale: $50K–$200K transactions are real (high-value legitimate and fraud).')
print('Clipping preserves the signal that high amounts are suspicious without distorting the scale.')

In [ ]:
# ---- Visualise amount distribution before and after winsorization --------
clean_df['amount_winsorized'] = clean_df['amount'].clip(upper=p99)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, title in zip(
    axes,
    ['amount', 'amount_winsorized'],
    ['Raw amount (log scale)', f'Winsorized amount (capped at ${p99:,.0f})']
):
    data = clean_df[col].dropna()
    fraud_amounts  = clean_df.loc[clean_df['is_fraud']==1, col].dropna()
    legit_amounts  = clean_df.loc[clean_df['is_fraud']==0, col].dropna()
    ax.hist(legit_amounts, bins=60, alpha=0.65, color='#42A5F5', label='Legitimate', density=True)
    ax.hist(fraud_amounts, bins=60, alpha=0.65, color='#EF5350', label='Fraud', density=True)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Transaction amount ($)')
    ax.set_ylabel('Density')
    ax.legend()
    if col == 'amount':
        ax.set_xscale('log')

plt.suptitle('Amount Distribution: Before vs After Winsorization', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# Apply winsorization permanently
clean_df['amount'] = clean_df['amount_winsorized']
clean_df.drop(columns=['amount_winsorized'], inplace=True)
print(f'Winsorization applied. Max amount now: ${clean_df["amount"].max():,.2f}')

## Section 4: Train / Test Split (Stratified)

**Golden rule reminder:** We split here and the test set will **not be touched again** until Section 6 final evaluation.  
All preprocessing — imputation, scaling, encoding — will be fit **only on training data** inside the pipeline.

In [ ]:
NUMERIC_FEATURES = ['amount','credit_score','account_balance',
                    'num_transactions_7d','account_age_days',
                    'balance_was_missing']   # MNAR indicator
CAT_ORDINAL      = ['risk_level']            # has natural order: low < medium < high
CAT_NOMINAL      = ['device_type','merchant_category']  # no order
ALL_FEATURES     = NUMERIC_FEATURES + CAT_ORDINAL + CAT_NOMINAL
TARGET           = 'is_fraud'

X = clean_df[ALL_FEATURES]
y = clean_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print('=== Stratified 80/20 Train/Test Split ===')
print(f'Train: {len(X_train):,} rows | Fraud: {y_train.sum():,} ({y_train.mean()*100:.2f} %)')
print(f'Test : {len(X_test):,}  rows | Fraud: {y_test.sum():,}  ({y_test.mean()*100:.2f} %)')
print(f'Expected fraud rate: {y.mean()*100:.2f} %')
print()
print('Fraud rates match perfectly — stratification is working.')

## Section 5: Build the Preprocessing Pipeline

### Pipeline Architecture

```
                    ┌─────────────────────────────────────────┐
                    │         ColumnTransformer               │
                    │                                         │
  numeric ──────────┤  SimpleImputer(median)                  │
  (6 columns)       │  → RobustScaler                         │
                    │                                         │
  ordinal ──────────┤  SimpleImputer('most_frequent')         │
  (risk_level)      │  → OrdinalEncoder                       │
                    │     (low=0, medium=1, high=2)           │
                    │                                         │
  nominal ──────────┤  SimpleImputer('most_frequent')         │
  (device + merch.) │  → OneHotEncoder(handle_unknown='ignore')│
                    └─────────────┬───────────────────────────┘
                                  │
                    LogisticRegression(
                        class_weight='balanced',
                        max_iter=1000
                    )
```

**Why RobustScaler?** It uses the median and IQR rather than mean and standard deviation.  
Even after winsorization, `amount` still has a skewed distribution. RobustScaler is less affected by remaining skewness than StandardScaler.

In [ ]:
# ---- Numeric pipeline: impute then scale ---------------------------------
numeric_pipeline = SklearnPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
])

# ---- Ordinal pipeline: impute then encode (preserves order) -------------
ordinal_pipeline = SklearnPipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[['low','medium','high']],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )),
])

# ---- Nominal pipeline: impute then one-hot encode -----------------------
nominal_pipeline = SklearnPipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    )),
])

# ---- Combine with ColumnTransformer -------------------------------------
preprocessor = ColumnTransformer([
    ('numeric',  numeric_pipeline,  NUMERIC_FEATURES),
    ('ordinal',  ordinal_pipeline,  CAT_ORDINAL),
    ('nominal',  nominal_pipeline,  CAT_NOMINAL),
], remainder='drop')

# ---- Full pipeline -------------------------------------------------------
baseline_pipeline = SklearnPipeline([
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE
    )),
])

print('Pipeline constructed successfully.')
print()
print('Pipeline steps:')
for step_name, step_obj in baseline_pipeline.steps:
    print(f'  {step_name}: {type(step_obj).__name__}')

## Section 6: Evaluate Baseline Pipeline

We use **5-fold StratifiedKFold cross-validation on the training set only**.  
The test set remains untouched.

In [ ]:
# ---- 5-fold StratifiedKFold CV -------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = cross_validate(
    baseline_pipeline, X_train, y_train,
    cv=skf,
    scoring={
        'auc'      : 'roc_auc',
        'precision': 'precision',
        'recall'   : 'recall',
        'f1'       : 'f1',
    },
    return_train_score=False,
    n_jobs=-1
)

metrics_df = pd.DataFrame({
    'Fold'     : [f'Fold {i+1}' for i in range(5)] + ['MEAN','STD'],
    'AUC-ROC'  : list(cv_results['test_auc']) +
                 [np.mean(cv_results['test_auc']), np.std(cv_results['test_auc'])],
    'Precision': list(cv_results['test_precision']) +
                 [np.mean(cv_results['test_precision']), np.std(cv_results['test_precision'])],
    'Recall'   : list(cv_results['test_recall']) +
                 [np.mean(cv_results['test_recall']), np.std(cv_results['test_recall'])],
    'F1'       : list(cv_results['test_f1']) +
                 [np.mean(cv_results['test_f1']), np.std(cv_results['test_f1'])],
})

print('=== Baseline Pipeline — 5-Fold CV Results ===')
print(metrics_df.round(4).to_string(index=False))

In [ ]:
# ---- Fit on full training set, evaluate on test set ---------------------
baseline_pipeline.fit(X_train, y_train)
y_pred_proba = baseline_pipeline.predict_proba(X_test)[:, 1]
y_pred       = baseline_pipeline.predict(X_test)

test_auc = roc_auc_score(y_test, y_pred_proba)
print(f'\n=== Baseline Pipeline — FINAL Test Set Performance ===')
print(f'AUC-ROC   : {test_auc:.4f}')
print(f'Precision : {precision_score(y_test, y_pred):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'F1        : {f1_score(y_test, y_pred):.4f}')

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Legitimate','Fraud'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Baseline Pipeline — Confusion Matrix\n(Test Set, AUC = {test_auc:.4f})',
             fontsize=12)
plt.tight_layout()
plt.show()

TN, FP, FN, TP = cm.ravel()
print(f'\nTrue Positives  (fraud caught)   : {TP}')
print(f'False Negatives (fraud missed)   : {FN}')
print(f'False Positives (false alarms)   : {FP}')
print(f'True Negatives  (correctly clear): {TN}')

## Section 7: Test for Data Leakage

Data leakage is one of the most common and damaging mistakes in machine learning.  
It occurs when information from the test set (or future data) is used — directly or indirectly — during training.

### Two leakage scenarios we will test:

1. **Preprocessing outside the pipeline** (leaky): fit the scaler on all data (train + test), then split.
2. **Pipeline inside cross-validation** (correct): the pipeline is passed to `cross_val_score` — each fold fits preprocessing only on that fold's training data.

In [ ]:
# ---- LEAKY approach: pre-process on ALL data, then split ----------------
# Step 1: impute on full dataset (leaks test stats into training)
X_numeric_only = clean_df[NUMERIC_FEATURES].copy()
leaky_imputer  = SimpleImputer(strategy='median')
X_imputed_all  = leaky_imputer.fit_transform(X_numeric_only)  # FIT ON ALL DATA ← LEAK

leaky_scaler   = RobustScaler()
X_scaled_all   = leaky_scaler.fit_transform(X_imputed_all)    # FIT ON ALL DATA ← LEAK

y_all = clean_df[TARGET]
X_tr_leak, X_te_leak, y_tr_leak, y_te_leak = train_test_split(
    X_scaled_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE
)

leaky_clf = LogisticRegression(class_weight='balanced', max_iter=1000,
                               random_state=RANDOM_STATE)
leaky_clf.fit(X_tr_leak, y_tr_leak)
auc_leaky = roc_auc_score(y_te_leak, leaky_clf.predict_proba(X_te_leak)[:, 1])

# ---- CORRECT approach: pipeline passed to cross_val_score ---------------
# Only numeric pipeline for fair comparison
correct_pipe = SklearnPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
    ('clf',     LogisticRegression(class_weight='balanced', max_iter=1000,
                                   random_state=RANDOM_STATE))
])

cv_aucs_correct = cross_val_score(
    correct_pipe,
    clean_df[NUMERIC_FEATURES], y_all,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc'
)

print('=== Data Leakage Comparison ===')
print(f'LEAKY pipeline AUC   : {auc_leaky:.4f}  ← optimistically inflated')
print(f'CORRECT pipeline AUC : {cv_aucs_correct.mean():.4f} ± {cv_aucs_correct.std():.4f}')
print(f'AUC gap              : {auc_leaky - cv_aucs_correct.mean():.4f}')
print()
print('Why is the leaky pipeline higher?')
print('The scaler and imputer saw test data statistics during fit.')
print('The model was calibrated using test-set information — artificially boosting performance.')
print()
print('The gap looks small on this dataset because the test set is large.')
print('On small datasets or with more complex preprocessing, the gap can exceed 0.10 AUC.')

### Leakage Checklist

Before finalising any model, verify:

- [ ] All imputers, scalers, and encoders are fit **inside** the pipeline, not on raw data.
- [ ] The pipeline object (not pre-processed arrays) is passed to `cross_val_score`.
- [ ] Feature engineering (ratios, aggregations) is done inside a `FunctionTransformer` in the pipeline.
- [ ] SMOTE (if used) is applied **after** the train/test split, inside `imblearn.pipeline.Pipeline`.
- [ ] The test set was not used to pick hyperparameters, thresholds, or features.

---

## Section 8: Compare Preprocessing Choices

In [ ]:
# We compare 4 preprocessing strategies on the numeric features only (for simplicity)
# All evaluations use 5-fold StratifiedKFold on X_train / y_train

X_num_train = X_train[NUMERIC_FEATURES]
X_num_test  = X_test[NUMERIC_FEATURES]

skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

variants = {
    'Variant 1\n(No preprocessing)': SklearnPipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
        ('clf',     LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Variant 2\n(Mean + MinMax)': SklearnPipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler',  MinMaxScaler()),
        ('clf',     LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Variant 3\n(Median + Standard\n+ balanced)': SklearnPipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('clf',     LogisticRegression(class_weight='balanced', max_iter=1000,
                                       random_state=RANDOM_STATE))
    ]),
    'Variant 4\n(Robust + KNN\n+ balanced)': SklearnPipeline([
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler',  RobustScaler()),
        ('clf',     LogisticRegression(class_weight='balanced', max_iter=1000,
                                       random_state=RANDOM_STATE))
    ]),
}

variant_results = {}
for name, pipe in variants.items():
    aucs = cross_val_score(pipe, X_num_train, y_train, cv=skf5, scoring='roc_auc')
    variant_results[name] = {'mean': aucs.mean(), 'std': aucs.std(), 'scores': aucs}
    print(f'{name.replace(chr(10)," "):50s} AUC = {aucs.mean():.4f} ± {aucs.std():.4f}')

In [ ]:
# ---- Bar chart comparing all 4 variants ---------------------------------
names  = [n.replace('\n', ' ') for n in variant_results.keys()]
means  = [v['mean'] for v in variant_results.values()]
stds   = [v['std']  for v in variant_results.values()]
colors = ['#EF5350','#FB8C00','#42A5F5','#26A69A']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(names, means, yerr=stds, capsize=6,
              color=colors, edgecolor='black', alpha=0.85, linewidth=0.8)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('AUC-ROC (5-fold CV mean)')
ax.set_title('Preprocessing Variants — AUC-ROC Comparison\n(numeric features only, Logistic Regression)',
             fontsize=12)
for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.003,
            f'{mean:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.xticks(fontsize=9)
plt.tight_layout()
plt.show()

# ---- Confusion matrices for best vs worst variant -----------------------
best_name  = names[np.argmax(means)]
worst_name = names[np.argmin(means)]
best_pipe  = list(variants.values())[np.argmax(means)]
worst_pipe = list(variants.values())[np.argmin(means)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, pipe, title in zip(axes, [best_pipe, worst_pipe], [best_name, worst_name]):
    pipe.fit(X_num_train, y_train)
    preds = pipe.predict(X_num_test)
    cm    = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=['Legit','Fraud']).plot(ax=ax, cmap='Blues', colorbar=False)
    auc_v = roc_auc_score(y_test, pipe.predict_proba(X_num_test)[:,1])
    ax.set_title(f'{title}\nTest AUC={auc_v:.4f}', fontsize=10)
plt.suptitle('Best vs Worst Variant — Confusion Matrices', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Section 9: Address Class Imbalance with SMOTE

### The Imbalance Problem

With only 2 % fraud, a naive model that predicts "legitimate" for every transaction achieves 98 % accuracy — but catches zero fraud.  
We have two main tools to fix this:

| Method | How it works | Pros | Cons |
|---|---|---|---|
| `class_weight='balanced'` | Re-weights the loss function; gives 50× more weight to fraud errors | Fast, no new data created | Does not create new training examples |
| **SMOTE** | Generates synthetic fraud samples by interpolating between real fraud neighbours | Balances training distribution | Must be inside the pipeline to avoid leakage; can oversample noise |

**Critical:** SMOTE must be applied **after** the train/test split and **inside** the pipeline.  
If SMOTE is run on the full dataset before splitting, synthetic samples derived from test-set fraud cases end up in the training set — that is leakage.

In [ ]:
# ---- Strategy A: class_weight only (Variant 3 from above) ----------------
pipe_cw = SklearnPipeline([
    ('preprocessor', preprocessor),
    ('clf',          LogisticRegression(class_weight='balanced', max_iter=1000,
                                        random_state=RANDOM_STATE))
])

# ---- Strategy B: SMOTE only (no class_weight) ----------------------------
# Must use imblearn.pipeline.Pipeline to support SMOTE between steps
pipe_smote = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote',        SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.2)),
    ('clf',          LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

# ---- Strategy C: SMOTE + class_weight (combined) -------------------------
pipe_combined = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote',        SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.2)),
    ('clf',          LogisticRegression(class_weight='balanced', max_iter=1000,
                                        random_state=RANDOM_STATE))
])

strategy_results = {}
for name, pipe in [
    ('class_weight only',   pipe_cw),
    ('SMOTE only',          pipe_smote),
    ('SMOTE + class_weight', pipe_combined),
]:
    scores = cross_val_score(pipe, X_train, y_train, cv=skf5, scoring='roc_auc')
    strategy_results[name] = scores
    print(f'{name:25s}  AUC = {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# ---- Precision-Recall Curves for all three strategies -------------------
fig, ax = plt.subplots(figsize=(9, 6))

strategy_colors = {'class_weight only': '#42A5F5',
                   'SMOTE only':         '#EF5350',
                   'SMOTE + class_weight': '#26A69A'}

for name, pipe in [
    ('class_weight only',    pipe_cw),
    ('SMOTE only',           pipe_smote),
    ('SMOTE + class_weight', pipe_combined),
]:
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(rec, prec, linewidth=2, label=f'{name} (AP={ap:.3f})',
            color=strategy_colors[name])

ax.axhline(y=y_test.mean(), linestyle='--', color='grey',
           label=f'No-skill baseline ({y_test.mean():.3f})')
ax.set_xlabel('Recall (Fraud Caught %)', fontsize=12)
ax.set_ylabel('Precision (Of Alerts, % Real Fraud)', fontsize=12)
ax.set_title('Precision-Recall Curve — Imbalance Handling Strategies\n(Higher and to the right is better)', fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

print()
print('Interpretation:')
print('  - Average Precision (AP) = area under the P-R curve')
print('  - For fraud detection, recall is critical: missing fraud is costly')
print('  - SMOTE + class_weight is often the strongest because it balances both')
print('    the training distribution (SMOTE) and the loss function (class_weight)')

## Section 10: Summary — Full Experiment Results

In [ ]:
# ---- Final AUC on test set for all pipelines ----------------------------
all_experiments = []

experiments = [
    ('No preprocessing',           list(variants.values())[0],    X_num_train, X_num_test),
    ('Mean + MinMax',              list(variants.values())[1],    X_num_train, X_num_test),
    ('Median + Standard + Balanced', list(variants.values())[2],  X_num_train, X_num_test),
    ('Robust + KNN + Balanced',    list(variants.values())[3],    X_num_train, X_num_test),
    ('Full pipeline (baseline)',   baseline_pipeline,             X_train,     X_test),
    ('Full + class_weight',        pipe_cw,                       X_train,     X_test),
    ('Full + SMOTE',               pipe_smote,                    X_train,     X_test),
    ('Full + SMOTE + class_weight', pipe_combined,                X_train,     X_test),
]

for exp_name, pipe, Xtr, Xte in experiments:
    pipe.fit(Xtr, y_train)
    proba = pipe.predict_proba(Xte)[:, 1]
    preds = pipe.predict(Xte)
    all_experiments.append({
        'Experiment'  : exp_name,
        'AUC-ROC'     : round(roc_auc_score(y_test, proba), 4),
        'Precision'   : round(precision_score(y_test, preds, zero_division=0), 4),
        'Recall'      : round(recall_score(y_test, preds, zero_division=0), 4),
        'F1'          : round(f1_score(y_test, preds, zero_division=0), 4),
    })

results_df = pd.DataFrame(all_experiments)
results_df_sorted = results_df.sort_values('AUC-ROC', ascending=False)
print('=== Final Experiment Summary (sorted by AUC-ROC) ===')
print(results_df_sorted.to_string(index=False))

In [ ]:
# ---- Final comparison plot -----------------------------------------------
fig, ax = plt.subplots(figsize=(12, 5))
exp_names   = results_df_sorted['Experiment'].tolist()
auc_values  = results_df_sorted['AUC-ROC'].tolist()
bar_colors  = ['#26A69A' if a == max(auc_values) else
               '#EF5350' if a == min(auc_values) else '#90CAF9'
               for a in auc_values]

bars = ax.barh(exp_names, auc_values, color=bar_colors,
               edgecolor='black', linewidth=0.7, alpha=0.85)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('AUC-ROC (Test Set)')
ax.set_title('All Experiments — Final AUC-ROC on Held-Out Test Set', fontsize=12, fontweight='bold')
ax.axvline(0.5, linestyle='--', color='red', alpha=0.4, label='Random baseline (0.5)')
for bar, val in zip(bars, auc_values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()

### Key Takeaways — What Preprocessing Choices Mattered Most?

| Finding | Impact |
|---|---|
| **Handling class imbalance** (`class_weight='balanced'` or SMOTE) | Largest single impact on recall — without it, the model ignores fraud entirely |
| **Using RobustScaler vs StandardScaler** | Moderate — Robust is more stable when `amount` is skewed even after winsorization |
| **KNN vs median imputation** | Small — both work, KNN marginally better at the cost of compute time |
| **OneHotEncoding merchant_category** | Moderate — adds signal from category-specific fraud patterns |
| **Adding the MNAR missingness flag** | Small but principled — preserves information about why balance data is missing |
| **Winsorizing amount** | Prevents extreme outliers from dominating the scale; important for non-robust scalers |

### What I Would Do Next

1. **Feature engineering:** 
   - Velocity features: count transactions in last 1h, 24h, 7d per user.
   - Ratio: `amount / account_balance` — unusually large relative spend.
   - Time-of-day bins: late-night transactions are more suspicious.
   - `days_since_last_transaction` per user.
2. **Better models:** Random Forest or Gradient Boosting (XGBoost, LightGBM) — tree models are less sensitive to scaling and handle imbalance well.
3. **Threshold tuning:** Instead of default 0.5 probability threshold, tune threshold on the validation set to optimise the business metric (e.g., maximise `recall` subject to `precision ≥ 30%`).
4. **Temporal validation:** Retrain on rolling windows to detect model drift as fraud patterns evolve.
5. **SHAP explainability:** Understand which features drive each prediction — essential for regulatory compliance in financial services.

## Section 11: Self-Check Questions

---

**Q1. The leaky pipeline got AUC 0.99 vs the correct pipeline's 0.84. What caused the gap?**

<details>
<summary>Show answer</summary>

The leaky pipeline fit its scaler and imputer on the **entire dataset** (including the test set) before splitting. This means the scaler learned the mean and variance of the test set, and the imputer learned the test-set median. When the model was then evaluated on the test set, its preprocessing was perfectly calibrated to those specific test examples — giving an unrealistically high AUC. In production, you would never have access to test-set statistics in advance, so the 0.99 AUC is a false promise. The correct pipeline (0.84) is the honest estimate.

</details>

---

**Q2. You used RobustScaler instead of StandardScaler. When would this matter for fraud data?**

<details>
<summary>Show answer</summary>

RobustScaler uses the median and IQR (interquartile range) rather than the mean and standard deviation. This matters when the data has:
- **Outliers** — extreme high-value transactions pull the mean and inflate the standard deviation, which squashes all non-outlier values into a tiny range after StandardScaler. RobustScaler is unaffected.
- **Skewed distributions** — `amount` follows an exponential-like distribution. Even after winsorization, the median is a better center than the mean.

In practice for fraud data: if even a few $200K transactions survived winsorization, StandardScaler would compress all normal transactions to near-zero scaled values. RobustScaler preserves the spread of the bulk of the data.

</details>

---

**Q3. SMOTE was applied inside the Pipeline before train/test split. Is this correct? Why?**

<details>
<summary>Show answer</summary>

No — this is **data leakage**. SMOTE generates synthetic minority samples by interpolating between real fraud examples. If SMOTE is applied before the train/test split, synthetic samples are created using information from the test set's fraud cases. Those synthetic samples then appear in the training set, meaning the model has indirectly trained on test-set patterns. The correct approach: always split first, then apply SMOTE only on the training fold, inside the `imblearn.pipeline.Pipeline`. This is what we did in Section 9 — the SMOTE step is inside the pipeline, which is passed to `cross_val_score`, ensuring SMOTE runs only within each training fold.

</details>

---

**Q4. You tuned the pipeline on the test set to maximise AUC. Your reported AUC is 0.93. What's wrong?**

<details>
<summary>Show answer</summary>

The test set was used as a selection mechanism — you chose the pipeline that achieved the highest AUC on the test set. By doing this across multiple experiments, you have implicitly "trained on" the test set. The AUC of 0.93 reflects not just genuine model quality but also the random luck of which pipeline happened to fit those specific test-set examples well. The true generalisation performance on new, unseen data will be lower. This is called **test-set overfitting** or **selection bias**. The fix: use a validation set (or cross-validation) for all tuning decisions; reserve the test set for the single final evaluation.

</details>

---

**Q5. Logistic regression with `class_weight='balanced'` vs SMOTE: which would you prefer and why?**

<details>
<summary>Show answer</summary>

Both are valid; the choice depends on context:

**Prefer `class_weight='balanced'` when:**
- The dataset is very large — SMOTE can be slow on millions of rows.
- You want a simple, fast baseline.
- You are using a model that natively supports `class_weight` (logistic regression, SVM, random forest).

**Prefer SMOTE when:**
- The minority class is extremely rare (< 1 %) and the model needs more positive examples to learn good decision boundaries.
- You are using a model that does not support class weighting (e.g., some neural network architectures).
- You have enough legitimate positives to generate high-quality synthetic neighbours (poor with very few real fraud cases).

**In practice:** combining both (`class_weight='balanced'` + SMOTE) often gives the best results because they address imbalance at two levels — the data distribution and the loss function. However, the improvements over `class_weight='balanced'` alone are often small for logistic regression. Gradient boosting with `scale_pos_weight` often outperforms both on fraud data.

</details>